# 03 Replication Results

Phase 6 notebook for Table-3 equivalent metrics and portfolio visualizations.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED_DIR

processed = Path(PROCESSED_DIR)
r2 = pd.read_csv(processed / 'eval_oos_r2_latest.csv', index_col=0).sort_values('oos_r2', ascending=False)
ic = pd.read_csv(processed / 'eval_ic_stats_latest.csv', index_col=0)
perf = pd.read_csv(processed / 'eval_portfolio_perf_latest.csv', index_col=0)
print('Top OOS R2 models')
print(r2.head(10).to_string())
print('\nTop Sharpe models')
print(perf.sort_values('sharpe', ascending=False).head(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
r2['oos_r2'].sort_values(ascending=False).plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_title('Phase 4 Pooled OOS R2 by Model')
ax.set_ylabel('OOS R2')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
perf['sharpe'].sort_values(ascending=False).plot(kind='bar', ax=ax, color='#ff7f0e')
ax.set_title('Phase 4 Long-Short Sharpe by Model')
ax.set_ylabel('Sharpe')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
portfolio_path = processed / 'portfolio_returns.parquet'
if portfolio_path.exists():
    port = pd.read_parquet(portfolio_path)
    port['date'] = pd.to_datetime(port['date'])
    if 'ls_ret' in port.columns:
        ls = port[['model', 'date', 'ls_ret']].copy()
    elif set(['model', 'date', 'long_ret', 'short_ret']).issubset(port.columns):
        ls = port[['model', 'date', 'long_ret', 'short_ret']].copy()
        ls['ls_ret'] = ls['long_ret'] - ls['short_ret']
    else:
        ls = None
else:
    ls = None

if ls is None:
    print('portfolio_returns.parquet with ls_ret not available. Skipping cumulative plot.')
else:
    top3 = perf['sharpe'].sort_values(ascending=False).head(3).index.tolist()
    fig, ax = plt.subplots(figsize=(10, 4))
    for m in top3:
        s = ls.loc[ls['model'] == m, ['date', 'ls_ret']].sort_values('date').set_index('date')['ls_ret']
        cum = (1 + s).cumprod() - 1
        ax.plot(cum.index, cum.values, label=m)
    ax.set_title('Cumulative Long-Short Returns (Top-3 Sharpe Models)')
    ax.set_ylabel('Cumulative return')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()